# How much peak relief can flexibility provide?

Independent decision study by Abdulaziz Aldoseri. Explore an additional ideal store's daily peak relief, then inspect the gap between forecast scheduling and hindsight potential.

The displayed forecast LP is the frozen solver-selected reference. Multiple schedules can share its best forecast peak and minimum throughput. A separate, explicitly post-hoc diagnostic below shows that selection among those ties materially affects outturn results. It does not replace the original policy or establish a validated improvement.

**Supported by National Energy SO Open Data.** Historic Demand Data 2024/2025, retrieved 14 September 2026; processing and model by Abdulaziz Aldoseri. [Source](https://www.neso.energy/data-portal/historic-demand-data) · [National Energy SO Open Data Licence v1.0](https://www.neso.energy/data-portal/neso-open-licence).

## Open the complete study

Run the cells in order in a fresh CPU runtime. Setup automatically retrieves the complete, frozen study package and verifies its exact size and SHA-256 before extracting or importing project code. Data, model code, source attribution, tests and full evaluated results are included; no separate file download, upload or Drive access is needed. Prepared or aggregate data retains the scope documented below and in the source register; it is not a claim that every publisher's raw export is redistributed.

Google Colab setup installs this study's pinned requirements where required. For local Jupyter, install the package requirements first. The acquisition and extraction path has been tested locally, including corruption, retrieval failure and rerun checks. Hosted Colab execution has not been verified. An unpublished website preview's Colab link becomes available when the notebook is published to GitHub.

This delivery edition changes setup only. The original scientific notebook, code, data, evaluation protocol and results remain in the unchanged reproduction package. The setup cell exposes the exact package URL and checksum for inspection.


In [ ]:
"""Validate and unpack the supplied project ZIP for a fresh Colab session."""
from pathlib import Path, PurePosixPath
import hashlib
import io
import json
import stat
import zipfile


def prepare_colab_archive(payload, destination):
    """Verify the package manifest, then extract only inside destination.

    The manifest detects corruption; it does not authenticate the publisher.
    The delivery setup verifies the pinned archive before calling this helper.
    Existing identical files are retained; differing files are never overwritten.
    """
    destination = Path(destination)
    if destination.is_symlink():
        raise ValueError('The project destination must not be a symbolic link.')
    root = destination.resolve()
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        infos = archive.infolist()
        if len(infos) > 1000 or sum(info.file_size for info in infos) > 100_000_000:
            raise ValueError('Archive exceeds this small project\'s extraction limits.')
        names = [info.filename for info in infos]
        if len(names) != len(set(names)):
            raise ValueError('Archive contains duplicate paths.')
        for info in infos:
            path = PurePosixPath(info.filename)
            if (path.is_absolute() or path.as_posix() != info.filename or '..' in path.parts or '\\' in info.filename
                    or ':' in info.filename or len(path.parts) < 2
                    or path.parts[0] != 'energy-flexibility'
                    or stat.S_ISLNK(info.external_attr >> 16) or info.is_dir()):
                raise ValueError('Archive contains an unexpected or unsafe path.')
        manifest_name = 'energy-flexibility/PACKAGE_MANIFEST.json'
        manifest = json.loads(archive.read(manifest_name))
        expected = {'energy-flexibility/' + row['path']: row for row in manifest}
        if len(expected) != len(manifest) or set(names) != set(expected) | {manifest_name}:
            raise ValueError('Archive contents do not match the package manifest.')
        required = {'energy-flexibility/energy_model.py', 'energy-flexibility/pipeline.py',
                    'energy-flexibility/requirements.txt',
                    'energy-flexibility/data/historic_demand_data_2024.csv',
                    'energy-flexibility/data/historic_demand_data_2025.csv'}
        if not required.issubset(expected):
            raise ValueError('The retrieved package is incomplete or is not the energy study.')
        checked = {}
        for name in names:
            data = archive.read(name)
            if name != manifest_name:
                row = expected[name]
                if len(data) != row['bytes'] or hashlib.sha256(data).hexdigest() != row['sha256']:
                    raise ValueError('A packaged file failed checksum verification.')
            relative = PurePosixPath(name).relative_to('energy-flexibility')
            target = root.joinpath(*relative.parts)
            if not target.resolve().is_relative_to(root) or target.is_symlink():
                raise ValueError('Extraction would leave the project directory.')
            if target in checked:
                raise ValueError('Archive paths resolve to the same destination.')
            if target.exists() and (not target.is_file() or target.read_bytes() != data):
                raise ValueError('The destination contains different files. Use a fresh Colab runtime.')
            checked[target] = data
        for target in checked:
            for parent in target.parents:
                if parent == root:
                    break
                if parent in checked or (parent.exists() and not parent.is_dir()):
                    raise ValueError('An archive file conflicts with a required directory.')
        if root.exists() and not root.is_dir():
            raise ValueError('The project destination is not a directory.')
        # Every archive member and destination has been checked before any write.
        root.mkdir(parents=True, exist_ok=True)
        for target, data in checked.items():
            target.parent.mkdir(parents=True, exist_ok=True)
            if not target.exists():
                target.write_bytes(data)
    return root



# Delivery-only setup. The scientific package remains the frozen original.
import os
import tempfile
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

STUDY_ID = 'energy'
PACKAGE_URL = 'https://raw.githubusercontent.com/Abdulaziz-Aldoseri/abdulaziz-aldoseri.github.io/master/public/files/energy/energy-flexibility-reproducibility.zip'
PACKAGE_SHA256 = 'd17cfc20507db6985aba37ae6c17a9c592b5c3387a69e3932eaffbdd85f0e424'
PACKAGE_BYTES = 3722765


def load_study_package(package_url=PACKAGE_URL, destination=None):
    """Fetch the complete pinned package; verify before extraction or imports.

    The optional URL/directory arguments permit local delivery verification.
    Analysis inputs and expected archive bytes are never inferred from a URL.
    """
    from hashlib import sha256
    try:
        with urlopen(package_url, timeout=30) as response:
            payload = response.read(PACKAGE_BYTES + 1)
    except (HTTPError, URLError, TimeoutError, OSError) as error:
        raise RuntimeError(
            "The study package could not be retrieved. Check the connection and "
            "rerun this cell. An unpublished preview becomes available after its "
            "GitHub release. No study code has been imported."
        ) from error
    if len(payload) != PACKAGE_BYTES or sha256(payload).hexdigest() != PACKAGE_SHA256:
        raise ValueError("The study package failed its pinned size/SHA-256 check. Nothing was extracted.")
    target = Path(destination) if destination is not None else (
        Path(tempfile.gettempdir()) / ("decision-lab-" + STUDY_ID + "-" + PACKAGE_SHA256[:16])
    )
    if target.exists():
        # Bind cached verification to the freshly pinned archive, never to a
        # mutable local manifest. Reject extra modules that could shadow imports.
        from io import BytesIO
        from zipfile import ZipFile
        if target.is_symlink() or not target.is_dir():
            raise ValueError("The cached study directory is unsafe.")
        prefix = "energy-flexibility/" if STUDY_ID == "energy" else ""
        manifest_name = "PACKAGE_MANIFEST.json" if STUDY_ID == "energy" else "package_manifest.json"
        with ZipFile(BytesIO(payload)) as pinned:
            trusted_manifest = pinned.read(prefix + manifest_name)
            allowed_roots = {name[len(prefix):].split("/", 1)[0] for name in pinned.namelist()}
        local_manifest = target / manifest_name
        if local_manifest.is_symlink() or not local_manifest.is_file() or local_manifest.read_bytes() != trusted_manifest:
            raise ValueError("The cached manifest differs from the pinned study package.")
        allowed_generated = {"__pycache__"}
        if STUDY_ID == "airline":
            allowed_generated.add("notebook_reproduction")
        if STUDY_ID == "mobility":
            allowed_generated.update({"local-source-downloads", "reproduced-aggregates"})
        for entry in target.iterdir():
            if entry.is_symlink():
                raise ValueError("A cached study entry is a symlink.")
            working_notebook = entry.is_file() and entry.suffix.lower() == ".ipynb"
            if entry.name not in allowed_roots | allowed_generated and not working_notebook:
                raise ValueError("Unexpected cached study entry could shadow code: " + entry.name)
    return prepare_colab_archive(payload, target)

# Start the study
import importlib.util
import subprocess
import sys
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except (ImportError, ModuleNotFoundError):
    IN_COLAB = False
PROJECT = load_study_package()
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT / "requirements.txt")], check=True)
    for package, expected_version in [("numpy", "2.3.5"), ("scipy", "1.16.3")]:
        already_loaded = sys.modules.get(package)
        if already_loaded is not None and getattr(already_loaded, "__version__", None) != expected_version:
            raise RuntimeError("A dependency was already loaded at another version. Restart the Colab runtime and rerun setup.")
os.chdir(PROJECT)
print("Complete study verified and ready. Continue with the analysis cells below.")


In [ ]:
from pathlib import Path
from datetime import date
import json
import sys

ROOT = Path.cwd()
if not (ROOT / 'energy_model.py').is_file():
    raise RuntimeError('Start the notebook from the energy-flexibility project folder.')
sys.path.insert(0, str(ROOT))
import energy_model as model
from pipeline import HASHES, evaluation_dates
import numpy as np

actual = model.load_demand(ROOT / 'data/historic_demand_data_2025.csv', HASHES[2025])
training = model.load_demand(ROOT / 'data/historic_demand_data_2024.csv', HASHES[2024])
profile = model.fit_forecast(training)
print(f'Verified {len(training)} training days and {len(evaluation_dates())} evaluation days.')


## Decision model

Minimize adjusted forecast peak `z`, subject to `forecast[t]+u[t]<=z`, `-P<=u[t]<=P`, `e[t]=e[t-1]+0.5*u[t]`, `0<=e[t]<=E` and `e[0]=e[T]=0`. Positive flow charges. The second LP minimizes `0.5*sum(abs(u))` while retaining the optimum peak within 0.00001 MW.

Remaining tied schedules are selected numerically by the pinned SciPy/HiGHS run. This is part of the frozen reference policy; equal forecast objectives do not imply equal realized outcomes.

This is a continuous LP. The store is hypothetical and lossless. The model is one node with no prices, real asset parameters or network constraints. ND is metered generation requirement, not total end-user demand. Hindsight uses outturn and is a perfect-information bound.

Change only the date and capacity values below to inspect another evaluated day. The forecast remains frozen from 2024.


In [ ]:
selected_day = date(2025, 11, 3)
power_mw = 1000
duration_h = 2
energy_capacity_mwh = power_mw * duration_h
if selected_day not in evaluation_dates():
    raise ValueError('Select a day between 1 May and 31 December 2025.')

forecast = model.predict_forecast(selected_day, profile)
forecast_solution = model.solve_schedule(forecast, power_mw, energy_capacity_mwh)
rule = model.forecast_window_rule(forecast, power_mw, energy_capacity_mwh)
hindsight = model.solve_schedule(actual[selected_day], power_mw, energy_capacity_mwh)
baseline_peak = max(actual[selected_day])
for name, solution in [('Forecast LP', forecast_solution), ('Forecast-window rule', rule), ('Hindsight bound', hindsight)]:
    realized_peak = max(actual[selected_day] + solution['flow_mw'])
    print(f'{name:24} peak {realized_peak:,.2f} MW; relief {baseline_peak-realized_peak:,.2f} MW; throughput {solution["throughput_mwh"]:,.2f} MWh')
print('Physical checks:', model.validate_schedule(forecast_solution['flow_mw'], power_mw, energy_capacity_mwh))


## Inspect the trajectory and clock changes

Every interval is half an hour. Autumn repetitions carry distinct UTC offsets and remain separate storage transitions. Energy has T+1 boundaries; `energy[t]` below is the state at the beginning of the interval.


In [ ]:
labels = model.settlement_labels(selected_day)
print('Period | London time | Actual MW | Forecast MW | Charge MW | Opening MWh')
for i, label in enumerate(labels):
    print(f'{i+1:6} | {label["local_time"]} {label["utc_offset"]} | {actual[selected_day][i]:9.1f} | {forecast[i]:11.1f} | {forecast_solution["flow_mw"][i]:9.1f} | {forecast_solution["energy_mwh"][i]:11.1f}')
print('Closing energy:', forecast_solution['energy_mwh'][-1])
print('Autumn day intervals:', len(model.settlement_labels(date(2025, 10, 26))))


## All-day evidence

The reference grid covers 0, 500, 1,000 and 2,000 MW at 1, 2 and 4 hours. The following figures use every test day. Negative relief is retained. A reached resource bound does not by itself prove that expansion has value.


In [ ]:
summary = json.loads((ROOT / 'outputs/summary.json').read_text(encoding='utf-8'))
for row in summary:
    if row['power_mw'] == power_mw and row['duration_h'] == duration_h:
        print(f'{row["policy"]:15} mean relief {row["mean_reduction_mw"]:8.2f} MW; worsened {row["worsened_days"]}/{row["days"]} days; worst {row["worst_reduction_mw"]:8.2f} MW')

saved_day = json.loads((ROOT / 'outputs/days' / f'{selected_day.isoformat()}.json').read_text(encoding='utf-8'))
scenario_key = f'{power_mw}_{duration_h}'
if scenario_key in saved_day['scenarios']:
    saved_schedule = saved_day['scenarios'][scenario_key]['policies']['forecast_lp']
    if np.allclose(forecast_solution['flow_mw'], saved_schedule['flow_mw'], atol=1e-5, rtol=0):
        print('Selected forecast schedule matches saved export.')
    else:
        np.testing.assert_allclose(forecast_solution['peak_mw'], saved_schedule['planned_peak_mw'], atol=2e-5, rtol=0)
        np.testing.assert_allclose(forecast_solution['throughput_mwh'], saved_schedule['throughput_mwh'], atol=2e-5, rtol=0)
        print('This environment selected a different tied schedule with the same two forecast objectives. The current rerun above may have different outturn results; the saved summary remains the frozen reference.')


## Post-hoc diagnostic: tied schedules

The following diagnostic was developed after inspecting the frozen results. It preserves the optimal forecast peak cap and original minimum throughput, then separately minimizes or maximizes `H=0.5*sum(e[t+1])`, the area under stored energy in MWh·h. Its schedule formulas use forecast and time only, but selecting between them after seeing these results would be test-period policy selection.

At 1,000 MW/two hours, maximum holding time gives 595.82 MW average relief and zero worsened days, compared with the frozen selection's 532.10 MW and 15 worsened days. Thus the 15-day finding describes one numerical selection, not every optimum of the forecast LP. No primary outputs have been changed. This is not a validated operational improvement; a justified tie preference needs development evidence and a fresh test period.


In [ ]:
diagnostic = json.loads((ROOT / 'diagnostics/tie_sensitivity.json').read_text(encoding='utf-8'))
print(diagnostic['classification'])
for name, row in diagnostic['time_based_tie_variants'].items():
    print(f'{name:20} mean relief {row["mean_mw"]:.2f} MW; worsened {row["worsened_days"]}/245 days')
print('Objective and physical checks:', diagnostic['max_absolute_objective_and_physical_residuals'])
print('Reproduce separately with: python diagnostics/tie_sensitivity.py')


## Interpretation

At 1,000 MW and two hours, the frozen forecast LP mean relief was 532.10 MW versus 514.41 MW for the rule. That particular LP selection worsened 15 of 245 peaks; the rule worsened none in this test set. Hindsight averaged 964.13 MW and uses unavailable future information. The tie diagnostic shows that the original comparison does not isolate forecast quality from numerical selection among optima. Neither the original average gain nor the post-hoc alternative establishes a reliable operational improvement.

The frozen forecast is an intentionally simple seasonal/daytype mean, with no weather or holiday inputs. Historic annual files are revised and are not complete as-issued vintages. Potential findings concern this ideal model, not real grid security, deployed equipment, savings or NESO endorsement. See `METHODS.md`, `RESULTS.md`, `data/README.md` and `QA_REVIEW.md` for the full record.
